In [0]:
%run "./BRONZE"


In [0]:
%run "./BRONZE"


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
import os
import pandas as pd 
import datetime as dt
from pyspark.sql import SparkSession

In [0]:
BASE_PATH = "abfss://data@rgemrhealthcare.dfs.core.windows.net/"

BRONZE_PATH = BASE_PATH + "bronze/"
SILVER_PATH = BASE_PATH + "silver/"

%md
## Clean Departments

In [0]:
silver_Departments=(
    departments
    .withColumn('DeptID',trim('DeptID'))
    .withColumn('Name',trim('Name'))
    .withColumn('Location',trim('Location'))
    .withColumn('Specialization',trim('Specialization'))
    .filter(col('DeptID').isNotNull())
    .dropDuplicates(['DeptID'])
)
silver_Departments.count()

%md
## Clean Providers

In [0]:
silver_Providers = (
    providers
    .withColumn("ProviderID", trim(col("ProviderID")))
    .withColumn("ProviderName", trim(col("ProviderName")))
    .withColumn("Specialization", trim(col("Specialization")))
    .withColumn("DeptID", trim(col("DeptID")))
    .withColumn("Phone", trim(col("Phone")))
    .withColumn("Email", trim(col("Email")))
    .withColumn("HireDate", trim(col("HireDate")))
    .filter((col("ProviderID").isNotNull()) &(col('DeptID').isNotNull()))
    .dropDuplicates(["ProviderID"])
)
silver_Providers.count()

%md
## Clean Patients

In [0]:
silver_patients=(

    patients
    .withColumn(('PatientID'),trim(col('PatientID')))
    .withColumn(('PatientName'),trim(col('PatientName')))
    .withColumn(('Gender'),trim(col('Gender')))
    .withColumn('DateOfBirth', to_date(col('DateOfBirth'), 'yyyy-MM-dd'))
    .withColumn(('Phone'),trim(col('Phone')))
    .withColumn(('Email'),trim(col('Email')))
    .withColumn(('Address'),trim(col('Address')))
    .withColumn(('BloodGroup'),trim(col('BloodGroup')))
    .withColumn(('InsuranceType'),trim(col('InsuranceType')))
    .withColumn(('RegistrationDate'),trim(col('RegistrationDate')))
    .filter(col('PatientID').isNotNull())
    .dropDuplicates(['PatientID'])
)
silver_patients.count()
display(silver_patients)

%md
## Clean ENCOUNTER

In [0]:
 silver_encounters=(
     encounters
     .withColumn('EncounterID',trim(col('EncounterID')))
     .withColumn('PatientID',trim(col('PatientID')))
     .withColumn('ProviderID',trim(col('ProviderID')))
     #.withColumn(('EncounterDate'),to_date(col('EncounterDate'),'MM/dd/yyyy'))
     .withColumn('EncounterDate', to_date(col('EncounterDate'), 'yyyy-MM-dd'))
     .withColumn('EncounterType',trim(col('EncounterType')))
     .withColumn('FollowUpDate', to_date(col('FollowUpDate'), 'yyyy-MM-dd'))
     .withColumn('Status',trim(col('Status')))
     .withColumn('Diagnosis',trim(col('Diagnosis')))
     .withColumn('TreatmentAdvice',trim('TreatmentAdvice'))
     .filter((col('EncounterID').isNotNull()) &(col('PatientID').isNotNull()) & (col('ProviderID').isNotNull()))
     .dropDuplicates(['EncounterID'])
 )
 silver_encounters.count()

 display(silver_encounters.limit(10))
 

%md
## Clean TRANSACTION

In [0]:
silver_transaction=(

    transactions
    .withColumn('TransactionID',trim(col('TransactionID')))
    .withColumn('EncounterID',trim(col('EncounterID')))
    .withColumn('TransactionDate',to_date(col('TransactionDate'),'yyyy-MM-dd'))
    .withColumn('TransactionType',trim(col('TransactionType')))
    .withColumn('Amount',col('Amount').cast('double'))
    .withColumn('PaymentMethod',trim(col('PaymentMethod')))
    .filter((col('TransactionID').isNotNull()) & (col('EncounterID').isNotNull()))
    .dropDuplicates(['TransactionID'])
)
silver_transaction.count()
display(silver_transaction.limit(10))


## 15. Referential validation


#### Encounters must have valid patients


In [0]:
silver_encounter_patients_join_df = (
    silver_encounters.alias('se')
    .join(silver_patients.alias('sp'), 'PatientID', 'inner')
    #.select('se.*')
    .select(
        col("se.EncounterID"),
        col("se.PatientID"),
        col("sp.PatientName"),
        col("sp.Gender"),
        col("sp.DateOfBirth"),
        col("sp.BloodGroup"),
        col("sp.InsuranceType"),
        col("se.ProviderID"),
        col("se.EncounterDate"),
        col("se.EncounterType"),
        col("se.Status"),
        col("se.Diagnosis"),
        col("se.TreatmentAdvice"),
        col("se.FollowUpDate")
))

silver_encounter_patients_join_df.count()
display(silver_encounter_patients_join_df.limit(10))

## Provider Having Valid Departments

In [0]:
silver_provider_dept_df = (
    silver_Providers.alias('sp')
    .join(silver_Departments.alias('sd'), 'DeptID', 'inner')
    .select(
        col("sp.ProviderID"),
        col("sp.ProviderName"),
        col("sp.Specialization").alias("ProviderSpecialization"),
        col("sp.DeptID"),
        col("sd.Name").alias("DepartmentName"),
        col("sd.Location").alias("DepartmentLocation"),
        col("sd.Specialization").alias("DepartmentSpecialization"),
        col("sp.Phone"),
        col("sp.Email"),
        col("sp.HireDate")
    )
)

silver_provider_dept_df.count()

## Multiple Join — Encounter + Patient + Provider + Department
#### Encounter--->Patinets
#### Encounter--->Provider
#### Provider---->Departments

In [0]:
silver_encounter_enriched_df = (
    silver_encounters.alias('e')
    .join(
        silver_patients.alias('p'),
        'PatientID',
        'left'
    )
    .join(
        silver_Providers.alias('pr'),
        'ProviderID',
        'left'
    )
    .join(
        silver_Departments.alias('d'),
        'DeptID',
        'left'
    )
    .select(
        col("e.EncounterID"),
        col("e.EncounterDate"),

        # Patient
        col("e.PatientID"),
        col("p.PatientName"),
        col("p.Gender"),
        col("p.DateOfBirth"),
        col("p.BloodGroup"),
        col("p.InsuranceType"),

        # Provider
        col("e.ProviderID"),
        col("pr.ProviderName"),
        col("pr.Specialization").alias("ProviderSpecialization"),

        # Department
        col("d.DeptID"),
        col("d.Name").alias("DepartmentName"),
        col("d.Location").alias("DepartmentLocation"),

        # Encounter
        col("e.EncounterType"),
        col("e.Status"),
        col("e.Diagnosis"),
        col("e.TreatmentAdvice"),
        col("e.FollowUpDate")
    )
)
encounter_enriched_df.count()
display(encounter_enriched_df.limit(10))

In [0]:
silver_transaction_encounter_df = (
    silver_transaction.alias("t")
    .join(
        silver_encounters.alias("e"),
        'EncounterID',
        "left"
    )
    .select(
        col("t.TransactionID"),
        col("t.EncounterID"),
        col("e.PatientID"),
        col("e.ProviderID"),
        col("e.EncounterDate"),
        col("e.EncounterType"),
        col("e.Diagnosis"),
        col("t.TransactionDate"),
        col("t.TransactionType"),
        col("t.Amount"),
        col("t.PaymentMethod"),
        col("e.Status"),
        ##col("t.InsuranceAmount"),
        #col("e.PatientAmount")
    )
)
display(silver_transaction_encounter_df.limit(10))

In [0]:
silver_encounter_enriched_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(SILVER_PATH + "encounter_enriched")
silver_transaction_encounter_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(SILVER_PATH + "transaction_encounter")
silver_provider_dept_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(SILVER_PATH + "provider_dept")

silver_encounter_patients_join_df.write\
    .format('delta')\
    .mode('overwrite')\
    .save(SILVER_PATH+ 'encounter_patients')
display(dbutils.fs.ls(SILVER_PATH))